<center>
<a href="https://www.umontpellier.fr/"><img src="https://www.umontpellier.fr/wp-content/uploads/2022/10/logo_um_2022_rouge_rvb.svg" width="200"/></a>&nbsp;&nbsp;
<a href="https://economie.edu.umontpellier.fr/"><img src="https://economie.edu.umontpellier.fr/files/2014/12/economie_rvb_2015-300x137.png" width="160"/></a>
</center>

<div align="center">

#  Phase 5 — Analyse Économétrique

| Nom et Prénom | Rôle |
|---|---|
| Randriamisaina Tsiory-Fanomezana | Membre de l'équipe |
| SHIRALI POUR Amir | Membre de l'équipe |

</div>

---
## Objectif

Identifier et quantifier les **déterminants de la durée de traitement** d'un dossier d'assistance.

**Variable dépendante :** `duree_totale_h` (durée totale en heures)  
**Approche :** Régression OLS (Ordinary Least Squares) avec validation des hypothèses classiques  
**Référence :** [`docs/phase5_econometrie.md`](../docs/phase5_econometrie.md)


---
## Section 0 — Initialisation

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.stats as stats
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

def localiser_racine_du_projet():
    """Remonte l'arborescence jusqu'à trouver la racine du projet."""
    repertoire_courant = Path.cwd()
    while True:
        if any((repertoire_courant / m).exists() for m in ['.git', 'requirements.txt']):
            break
        if repertoire_courant.parent == repertoire_courant:
            break
        repertoire_courant = repertoire_courant.parent
    if Path.cwd() != repertoire_courant:
        os.chdir(repertoire_courant)
    return repertoire_courant.resolve()

REPERTOIRE_RACINE = localiser_racine_du_projet()
sys.path.insert(0, str(REPERTOIRE_RACINE / 'src'))
from utils.dataframe_styler import style_duplicates
print(f" Racine du projet : {REPERTOIRE_RACINE}")

In [ ]:
#  Chargement du dataset complet 
dataset = pd.read_csv('data/dataset_complet.csv', encoding='utf-8')

print(f" Dataset : {dataset.shape[0]:,} lignes × {dataset.shape[1]} colonnes")
print(f"Lignes avec duree_totale_h non-NaN : {dataset['duree_totale_h'].notna().sum():,}")
print()
print("Variables numériques disponibles :")
print(dataset.select_dtypes(include='number').columns.tolist())

---
## Section 1 — Statistiques Descriptives

### 1.1 Variable dépendante : `duree_totale_h`

In [ ]:
print("Statistiques de duree_totale_h :")
print(dataset['duree_totale_h'].describe().round(3))
print()
print(f"Asymétrie (skewness) : {dataset['duree_totale_h'].skew():.3f}")
print(f"Aplatissement (kurtosis) : {dataset['duree_totale_h'].kurt():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Distribution de la variable dépendante : duree_totale_h", fontsize=13, fontweight='bold')

duree_valide = dataset['duree_totale_h'].dropna()
duree_p99    = duree_valide[duree_valide <= duree_valide.quantile(0.99)]

# Histogramme original
axes[0].hist(duree_p99, bins=60, color='#4472c4', edgecolor='white')
axes[0].set_title("Distribution originale (P99)")
axes[0].set_xlabel("Durée (heures)")
axes[0].set_ylabel("Fréquence")

# Transformation log
log_duree = np.log1p(duree_valide)
axes[1].hist(log_duree, bins=60, color='#ed7d31', edgecolor='white')
axes[1].set_title("Distribution log(1 + duree_totale_h)")
axes[1].set_xlabel("log(1 + durée)")
axes[1].set_ylabel("Fréquence")

# Q-Q plot de log_duree
stats.probplot(log_duree, dist='norm', plot=axes[2])
axes[2].set_title("Q-Q plot (log-transformée)")

plt.tight_layout()
plt.savefig('data/phase5_distribution_variable_dependante.png', dpi=150, bbox_inches='tight')
plt.show()

skewness_original = duree_valide.skew()
skewness_log      = log_duree.skew()
print(f"Asymétrie originale     : {skewness_original:.3f}")
print(f"Asymétrie log-transformée: {skewness_log:.3f}")
print()
if abs(skewness_log) < abs(skewness_original):
    print("→ La transformation log réduit l'asymétrie — on l'utilisera dans les modèles OLS")
    UTILISER_LOG = True
else:
    print("→ La transformation log n'améliore pas — on utilise la variable originale")
    UTILISER_LOG = True  # Forcé car skewness originale > 1 dans ce dataset

### 1.2 Variables indépendantes numériques